# Week 12 

**A complete Data Science project**

**I choose a start-up business problem.**

where early startup which as SaaS or a fintech app, doesn't having any historical data about frauds, like if that startup offers any subscriptions, fraudsters may use stolen credit cards(payment fraud), or if the company offers a promo code fraudster may create thouands of fake accounts to drain those promo codes.

**My objective is to flag such a payment frauds accounts without harming the legit user and also ensuring the company's captial won't slip into hands of fraudsters.**

*the dataset as decided.. the **PaySim Synthetic Dataset(from kaggle)***

In [17]:
import pandas as pd

df = pd.read_csv('data/paysim_sample.csv')

print(df.shape)

df.head()

(318131, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,10,CASH_IN,367088.24,C291409265,1328.10,368416.34,C1132709076,2312937.66,3837053.94,0,0
1,10,PAYMENT,27004.49,C535489359,0.00,0.00,M2110652315,0.00,0.00,0,0
2,8,PAYMENT,2173.75,C927615422,820.00,0.00,M1658216730,0.00,0.00,0,0
3,10,TRANSFER,19670.92,C1562020231,5416.73,0.00,C2060926688,63327.50,0.00,0,0
4,10,PAYMENT,17394.26,C1165434512,168543.14,151148.88,M490546299,0.00,0.00,0,0


#### **About Dataset**

***step***: unit of time. 1 step = 1 hour

***type***: CASH_IN, CASH_OUT, DEBT, PAYMENT and TRANSFER

***amount***: the charge or THE amount

***nameOrig***: customer who started the transaction

***oldbalanceOrg***: initial balance before the transaction 

***newbalanceOrig*** - new balance after the transaction.

***nameDest*** - customer who is the recipient of the transaction

***oldbalanceDest*** - initial balance recipient before the transaction. Note that there is not information for customers that start with M (Merchants).

***newbalanceDest*** - new balance recipient after the transaction. Note that there is not information for customers that start with M (Merchants).

***isFraud*** - This is the transactions made by the fraudulent agents inside the simulation. In this specific dataset the fraudulent behavior of the agents aims to profit by taking control or customers accounts and try to empty the funds by transferring to another account and then cashing out of the system.

***isFlaggedFraud*** - The business model aims to control massive transfers from one account to another and flags illegal attempts. An illegal attempt in this dataset is an attempt to transfer more than 200,000 in a single transaction.

In [18]:
#-----structural Overview-----
print("-----DataFrame info-----")
print(df.info())

#-----Checking for NULL values---
print("\n-----Missing values-----")
print(df.isna().sum())

-----DataFrame info-----
<class 'pandas.DataFrame'>
RangeIndex: 318131 entries, 0 to 318130
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   step            318131 non-null  int64  
 1   type            318131 non-null  str    
 2   amount          318131 non-null  float64
 3   nameOrig        318131 non-null  str    
 4   oldbalanceOrg   318131 non-null  float64
 5   newbalanceOrig  318131 non-null  float64
 6   nameDest        318131 non-null  str    
 7   oldbalanceDest  318131 non-null  float64
 8   newbalanceDest  318131 non-null  float64
 9   isFraud         318131 non-null  int64  
 10  isFlaggedFraud  318131 non-null  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 35.3 MB
None

-----Missing values-----
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud

In [21]:
df1=df.copy(deep=True)

### EDA

In [22]:
print("--- FRAUD CLASS DISTRIBUTION ---")
print(df1['isFraud'].value_counts())
print("\n--- FRAUD PERCENTAGE ---")
print(df1['isFraud'].value_counts(normalize=True) * 100)

--- FRAUD CLASS DISTRIBUTION ---
isFraud
0    317735
1       396
Name: count, dtype: int64

--- FRAUD PERCENTAGE ---
isFraud
0    99.875523
1     0.124477
Name: proportion, dtype: float64


In [26]:
print("--- TRANSACTION AMOUNTS: NORMAL VS FRAUD ---")
print(df1.groupby('isFraud')['amount'].describe())

--- TRANSACTION AMOUNTS: NORMAL VS FRAUD ---
            count          mean           std   min          25%         50%  \
isFraud                                                                        
0        317735.0  1.771326e+05  5.614648e+05  0.16   13318.6000   74675.350   
1           396.0  1.474271e+06  2.382247e+06  0.00  140689.1575  455223.335   

                 75%          max  
isFraud                            
0         208445.870  64234448.19  
1        1407085.135  10000000.00  


whoa.... a fruadster gain a max of 10M dollar... shock!!!

In [25]:
print("--- FRAUD BY TRANSACTION TYPE ---")
# Filter only fraudulent transactions, then count the 'type'
print(df1[df1['isFraud'] == 1]['type'].value_counts())

--- FRAUD BY TRANSACTION TYPE ---
type
CASH_OUT    203
TRANSFER    193
Name: count, dtype: int64


only in the two type , fine fine!.

encoding phase.

In [20]:
# pd.get_dummies(df, columns=['type'], drop_first=True, dtype=int)